# Dig into the SF calculations from Dan and X

In [1]:
# imports
from importlib import reload
import os

import xarray
import pandas

import numpy as np
# import fsspec
import matplotlib
import matplotlib.pyplot as plt
import gsw_xarray as gsw
from xhistogram.xarray import histogram

from profiler.loading.pymatreader import pymatreader

from strucFunct2_ai import timescale

from profiler import gliderdata
from profiler import profilerpairs
from cugn import io as cugn_io
from cugn import utils as cugn_utils
from cugn import plotting as cugn_plotting

import qg_utils
import strucFunct2_ai
import glider_io

# Load up

## Dan

In [2]:
idg_datafile = os.path.join(os.getenv('OS_SPRAY'), 'ARCTERX', 'Leg2', 'dr_2gliders.mat')

In [3]:
d = pymatreader.read_mat(idg_datafile)
d.keys()

dict_keys(['__header__', '__version__', '__globals__', 'lat', 'lon', 'missid', 'time', 'u', 'v', 'x', 'y'])

In [4]:
d['u'].shape

(100, 625)

## X

In [5]:
dataset = 'ARCTERX-2025'
profilers = glider_io.load_dataset(dataset)
profilers

Loading Sprays
calc_dist_offset: theta=1.5707963267948966 rad, 90.0 deg
calc_dist_offset: theta=1.5707963267948966 rad, 90.0 deg
Using lonendpts: (129.9167, 129.9167)
Using latendpts: (20.3332, 20.3334)
calc_dist_offset: theta=1.5707963267948966 rad, 90.0 deg
calc_dist_offset: theta=1.5707963267948966 rad, 90.0 deg


/home/xavier/Projects/Oceanography/python/profiler/profiler/profilerpairs.py:555: RuntimeWarning: Mean of empty slice
  avg_r.append(np.nanmean(self.r[in_r]))
/home/xavier/Projects/Oceanography/python/profiler/profiler/profilerpairs.py:556: RuntimeWarning: Mean of empty slice
  avg_S1.append(np.nanmean(self.S1[in_r]))
/home/xavier/miniconda3/envs/ocean/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/xavier/miniconda3/envs/ocean/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/xavier/Projects/Oceanography/python/profiler/profiler/profilerpairs.py:559: RuntimeWarning: Mean of empty slice
  avg_S2.append(np.nanmean(self.S2[in_r]))
/home/xavier/Projects/Oceanography/python/profiler/profiler/profilerpairs.py:561: RuntimeWarning: Mean of emp

[SprayData object for ARCTERX-Leg2
   Mission ID: 25203301
   Number of profiles: 306
   Time range: 2025-02-03 03:56:32.499998093 to 2025-04-10 22:56:39.250003338
   In field? False  ADCP on? True  Variables:
     time: (306,)
     lat: (306,)
     lon: (306,)
     s: (306, 100)
     t: (306, 100),
 SprayData object for ARCTERX-Leg2
   Mission ID: 25203801
   Number of profiles: 319
   Time range: 2025-02-03 03:50:20.749997616 to 2025-04-10 22:20:43.500003337
   In field? False  ADCP on? True  Variables:
     time: (319,)
     lat: (319,)
     lon: (319,)
     s: (319, 100)
     t: (319, 100)]

In [6]:
profilers[0].profile_id

# Construct pairs

In [7]:
max_time = 7.
gPairs = profilerpairs.ProfilerPairs(profilers, max_time=max_time, debug=False, randomize=True)

Using lonendpts: (np.float64(129.930225), np.float64(129.930225))
Using latendpts: (np.float64(20.365085), np.float64(20.365285))
calc_dist_offset: theta=1.5707963267948966 rad, 90.0 deg
calc_dist_offset: theta=1.5707963267948966 rad, 90.0 deg


In [8]:
gPairs

ProfilerPair object for the following datasets:
 ARCTERX-Leg2, SprayData 25203301 
 ARCTERX-Leg2, SprayData 25203801 
  Number of pairs: 912
  Time range: 2025-02-03 03:50:20.749997616 to 2025-04-10 22:56:39.250003338

## Print the IDs of the pairs

In [12]:
df = pandas.DataFrame()
df['missid_0'] = gPairs.data('missida', 0)
df['missid_1'] = gPairs.data('missida', 1)
#
df['profid_0'] = gPairs.data('profile_id', 0)
df['profid_1'] = gPairs.data('profile_id', 1)
# 
df.head()

,missid_0,missid_1,profid_0,profid_1
0,25203801,25203301,45,24
1,25203301,25203801,24,46
2,25203301,25203801,24,47
3,25203801,25203301,48,24
4,25203301,25203801,25,46


In [13]:
df.to_csv('pair_IDs.csv')